# Bài 17: Docker
## Đóng gói ứng dụng để chạy ở bất kỳ đâu

**Mục tiêu:**
- Hiểu tại sao cần Docker
- Viết `Dockerfile` cho Streamlit app
- Build và chạy container

---
## Phần 1: Lý thuyết

### Vấn đề không có Docker

Bạn gửi code cho người khác → họ chạy bị lỗi:
```
ModuleNotFoundError: No module named 'chromadb'
```
Vì máy họ thiếu thư viện, hoặc Python version khác.

---

### Docker là gì?

Docker tạo ra một **container** — môi trường cô lập chứa:
- Python đúng version
- Tất cả thư viện từ `requirements.txt`
- Code của bạn

Container chạy giống nhau trên mọi máy: Windows, Mac, Linux, cloud server.

```
Dockerfile  →  docker build  →  Image  →  docker run  →  Container
(công thức)    (nấu ăn)        (món ăn)   (dọn ra)       (đang ăn)
```

---

### Dockerfile — từng dòng

```dockerfile
FROM python:3.11-slim          # base image: Python 3.11 bản nhỏ gọn

WORKDIR /app                   # thư mục làm việc bên trong container

COPY requirements.txt .        # copy file vào container trước
RUN pip install -r requirements.txt  # cài thư viện (layer được cache)

COPY phase4/ ./phase4/         # copy code

EXPOSE 8501                    # khai báo port Streamlit dùng

CMD ["streamlit", "run",       # lệnh chạy khi container khởi động
     "phase4/app.py",
     "--server.address", "0.0.0.0"]
```

**Tại sao COPY requirements.txt trước rồi mới COPY code?**
Docker cache từng layer. Nếu code thay đổi nhưng requirements không đổi → Docker dùng lại layer `pip install` → build nhanh hơn nhiều.

---
## Phần 2: Bài tập

File `Dockerfile` và `.dockerignore` đã được tạo sẵn ở thư mục gốc project.

### Bước 1: Kiểm tra Docker đã cài chưa

Chạy trong terminal:

In [1]:
# Kiểm tra Docker version
import subprocess
result = subprocess.run(["docker", "--version"], capture_output=True, text=True)
print(result.stdout or "Docker chưa cài — tải tại https://docs.docker.com/get-docker/")

Docker version 27.0.3, build 7d4bcd8



### Bước 2: Build image

Chạy từ **thư mục gốc của project** (không phải trong `phase4/`):

```bash
docker build -t stock-assistant .
```

- `-t stock-assistant` — đặt tên cho image
- `.` — dùng Dockerfile ở thư mục hiện tại

### Bước 3: Chạy container

```bash
docker run -p 8501:8501 \
  -e GOOGLE_API_KEY=your_key_here \
  -v "$(pwd)/phase4/chroma_db:/app/phase4/chroma_db" \
  stock-assistant
```

- `-p 8501:8501` — map port container ra máy host
- `-e GOOGLE_API_KEY=...` — truyền API key qua biến môi trường (không COPY `.env` vào image)
- `-v` — mount thư mục `chroma_db` từ máy vào container (data không bị mất khi container dừng)

Mở trình duyệt: http://localhost:8501

### Dừng container

```bash
docker ps                    # xem container đang chạy
docker stop <container_id>   # dừng
```

---
## Tóm tắt

| File | Mục đích |
|------|----------|
| `Dockerfile` | Công thức build image |
| `.dockerignore` | Danh sách file không COPY vào image (giống `.gitignore`) |
| `docker build` | Tạo image từ Dockerfile |
| `docker run` | Khởi động container từ image |
| `-v` (volume) | Mount data từ máy host vào container — data persist khi container stop |
| `-e` (env var) | Truyền secrets mà không cần COPY `.env` vào image |